In [5]:
from datasets import load_dataset

dataset = load_dataset("wandb/RAGTruth-processed")
import pandas as pd

from hallucination_detector.data import clean, split_stratified

df = pd.DataFrame(dataset["train"])
clean_df = clean(df)
train, val = split_stratified(clean_df)

Repo card metadata block was not found. Setting CardData to empty.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from hallucination_detector.features import build_features
from pathlib import Path

vectorizer = TfidfVectorizer().fit(pd.concat([train["output"], train["context"]]))
train_path = Path("../data/X_train.csv")
val_path = Path("../data/X_val.csv")

if train_path.exists() and val_path.exists():
    X_train = pd.read_csv(train_path, index_col=0)
    X_val = pd.read_csv(val_path, index_col=0)
else:
    X_train = build_features(train, vectorizer)
    X_val = build_features(val, vectorizer)
    Path("../data").mkdir(exist_ok=True)
    X_train.to_csv(train_path)
    X_val.to_csv(val_path)

assert (X_train.index == train.index).all()

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, make_scorer
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

y_train = train["label"]
groups = train["context"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
f2 = make_scorer(fbeta_score, beta=2)

model = LogisticRegression(class_weight = "balanced", max_iter = 1000)
scores = cross_val_score(model, X_train, y_train, groups= groups, cv=cv, scoring=f2)
roc_scores = cross_val_score(model, X_train, y_train, groups= groups, cv=cv, scoring="roc_auc")

print(scores)
print(roc_scores)
print("mean", scores.mean(), "std", scores.std())
print("mean_roc", roc_scores.mean(), "std_roc", roc_scores.std())

[0.66367868 0.69904404 0.64016884 0.64830952 0.65934066]
[0.73295759 0.74643229 0.74606929 0.72742951 0.74630202]
mean 0.662108345802757 std 0.02022929323168325
mean_roc 0.7398381392838885 std_roc 0.00806731420386633


In [12]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000),
)
scores_roc = cross_val_score(model, X_train, y_train, groups=groups, cv=cv, scoring="roc_auc")
print("mean_roc", scores_roc.mean(), "std_roc", scores_roc.std())

mean_roc 0.7399145229244687 std_roc 0.008085089329777126


In [13]:
model.fit(X_train, y_train)
weights = model[-1].coef_[0]
print(pd.Series(weights, index=X_train.columns).sort_values())

word_overlap      -0.574750
tfidf_cosine      -0.367707
missing_numbers    0.124544
entity_overlap     0.245073
length_ratio       0.316200
dtype: float64


In [14]:
X_train.groupby(y_train).mean()

,word_overlap,length_ratio,missing_numbers,entity_overlap,tfidf_cosine
label,,,,,
0,0.757483,0.380089,0.139368,0.779404,0.583087
1,0.641661,0.464371,0.173172,0.768477,0.457685


In [15]:
for C in [0.0001, 0.001, 0.01, 0.1, 1, 10]:
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, class_weight="balanced", max_iter=1000),
    )
    roc = cross_val_score(model, X_train, y_train, groups=groups, cv=cv, scoring="roc_auc")
    model.fit(X_train, y_train)
    print(C, round(roc.mean(), 4), model[-1].coef_[0].round(3))

0.0001 0.7423 [-0.157  0.073  0.026  0.007 -0.131]
0.001 0.7417 [-0.415  0.232  0.081  0.119 -0.321]
0.01 0.7402 [-0.545  0.307  0.118  0.223 -0.369]
0.1 0.7399 [-0.572  0.315  0.124  0.243 -0.368]
1 0.7399 [-0.575  0.316  0.125  0.245 -0.368]
10 0.7399 [-0.575  0.316  0.125  0.245 -0.368]


In [16]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
gb_roc = cross_val_score(gb, X_train, y_train, groups=groups, cv=cv, scoring="roc_auc")
print(gb_roc)
print("mean", gb_roc.mean(), "std", gb_roc.std())

[0.7789753  0.78737748 0.78566912 0.77527927 0.78324224]
mean 0.7821086835401863 std 0.004431269404358866


In [17]:
from sklearn.metrics import roc_auc_score

gb.fit(X_train, y_train)
train_probs = gb.predict_proba(X_train)[:, 1]
print("train roc", roc_auc_score(y_train, train_probs))

train roc 0.830887523032089


In [18]:
import mlflow

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("ticket-6-features")


with mlflow.start_run(run_name="gradient_boosting"):
    mlflow.log_param("model", "HistGradientBoosting")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_splits", 5)
    mlflow.log_param("features", list(X_train.columns))
    mlflow.log_metric("cv_roc_auc_mean", gb_roc.mean())
    mlflow.log_metric("cv_roc_auc_std", gb_roc.std())

2026/11/20 16:12:35 INFO mlflow.tracking.fluent: Experiment with name 'ticket-6-features' does not exist. Creating a new experiment.


In [19]:

with mlflow.start_run(run_name="logistic_regression"):
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("C", 0.1)
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_splits", 5)
    mlflow.log_param("features", list(X_train.columns))
    mlflow.log_metric("cv_roc_auc_mean", scores_roc.mean())
    mlflow.log_metric("cv_roc_auc_std", scores_roc.std())